In [87]:
# Read the WhatsApp chat file

with open("my_chat.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total lines in file:", len(lines))
print(lines[0])
print(lines[1])
print(lines[2])

messages = []

participants = set()

system_count = 0
media_count = 0
deleted_count = 0

Total lines in file: 2939


[24/07/25, 8:40:34 PM] Kunal Sgt: ‎Kunal Sgt created this group

[14/02/26, 1:33:28 PM] Game: ‎Kunal Sgt added you



In [88]:
def parse_chat(lines):

    messages = []
    participants = set()

    system_count = 0
    media_count = 0
    deleted_count = 0

    current_message = None

    for line in lines:

        line = line.strip()

        if line == "":
            continue

        # New WhatsApp message starts with '['
        if line.startswith("["):

            # Save previous message
            if current_message:
                messages.append(current_message)

            # Remove opening '['
            line = line[1:]

            # Split timestamp from remaining text
            if "] " not in line:
                continue

            timestamp, rest = line.split("] ", 1)

            # System message
            if ": " not in rest:
                system_count += 1
                current_message = None
                continue

            sender, message = rest.split(": ", 1)

            participants.add(sender)

            # Separate date and time
            date, time = timestamp.split(", ", 1)

            if "<Media omitted>" in message:
                media_count += 1

            if "This message was deleted" in message:
                deleted_count += 1

            current_message = {
                "date": date,
                "time": time,
                "sender": sender,
                "message": message
            }

        else:
            # Multiline message
            if current_message:
                current_message["message"] += "\n" + line

    # Save last message
    if current_message:
        messages.append(current_message)

    return (
        messages,
        participants,
        system_count,
        media_count,
        deleted_count
    )

In [89]:
messages, participants, system_count, media_count, deleted_count = parse_chat(lines)

print(messages[0])

print(messages[1])

print(messages[2])

{'date': '24/07/25', 'time': '8:40:34\u202fPM', 'sender': 'Kunal Sgt', 'message': '\u200eKunal Sgt created this group'}
{'date': '14/02/26', 'time': '1:33:28\u202fPM', 'sender': 'Game', 'message': '\u200eKunal Sgt added you'}
{'date': '14/02/26', 'time': '1:39:09\u202fPM', 'sender': '~HarshS Sgt', 'message': 'Ye kaha exit krdiya group'}


In [91]:
days = set()

for msg in messages:
    days.add(msg["date"])

print("=" * 60)
print("FEATURE 1 : CHAT PARSER")
print("=" * 60)

print("Successfully parsed", len(messages), "messages")
print("Participants      :", len(participants))
print("Days Covered      :", len(days))
print("System Messages   :", system_count)
print("Media Omitted     :", media_count)
print("Deleted Messages  :", deleted_count)

print("=" * 60)

FEATURE 1 : CHAT PARSER
Successfully parsed 2833 messages
Participants      : 5
Days Covered      : 88
System Messages   : 1
Media Omitted     : 0
Deleted Messages  : 2


In [92]:
from datetime import datetime
def group_overview(messages):

    total_messages = len(messages)

    participants = set()

    message_count = {}

    dates = []

    # Count messages and collect dates
    for msg in messages:

        participants.add(msg["sender"])

        sender = msg["sender"]

        if sender in message_count:
            message_count[sender] += 1
        else:
            message_count[sender] = 1

        dates.append(datetime.strptime(msg["date"], "%d/%m/%y"))

    # Date range
    start_date = min(dates)
    end_date = max(dates)

    total_days = (end_date - start_date).days + 1

    # Sort by highest messages
    sorted_members = sorted(
        message_count.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Print Report
    print("=" * 60)
    print("GROUP OVERVIEW")
    print("=" * 60)

    print("Group            : Hostel Bois 4ever")
    print()

    print(
        f"Period           : {start_date.strftime('%d %B %Y')} "
        f"to {end_date.strftime('%d %B %Y')} ({total_days} days)"
    )

    print(f"Total Messages   : {total_messages:,}")
    print(f"Participants     : {len(participants)}")

    print()
    print("MESSAGES PER PERSON")
    print("-" * 60)

    for name, count in sorted_members:

        percentage = (count / total_messages) * 100

        print(f"{name:<10} : {count:>4} ({percentage:5.1f}%)")

In [93]:
group_overview(messages)

GROUP OVERVIEW
Group            : Hostel Bois 4ever

Period           : 24 July 2025 to 29 June 2026 (341 days)
Total Messages   : 2,833
Participants     : 5

MESSAGES PER PERSON
------------------------------------------------------------
~HarshS Sgt :  944 ( 33.3%)
Kunal Sgt  :  739 ( 26.1%)
Naman 2    :  676 ( 23.9%)
Rahul      :  473 ( 16.7%)
Game       :    1 (  0.0%)


In [94]:
# feature 3
from datetime import datetime

def busiest_day_hour(messages):

    day_count = {}
    hour_count = {}

    # Count messages per day and hour
    for msg in messages:

        date = msg["date"]
        hour = int(msg["time"].split(":")[0])

        if date in day_count:
            day_count[date] += 1
        else:
            day_count[date] = 1

        if hour in hour_count:
            hour_count[hour] += 1
        else:
            hour_count[hour] = 1

    # Find busiest day
    busiest_day = max(day_count, key=day_count.get)
    busiest_day_messages = day_count[busiest_day]

    # Find busiest hour
    busiest_hour = max(hour_count, key=hour_count.get)
    busiest_hour_messages = hour_count[busiest_hour]

    # Total unique days
    total_days = len(day_count)

    avg_messages = busiest_hour_messages / total_days

    # Format date
    formatted_day = datetime.strptime(
        busiest_day,
        "%d/%m/%y"
    ).strftime("%d %B %Y")

    print("=" * 60)
    print("MOST ACTIVE DAY AND HOUR")
    print("=" * 60)

    print(f"Busiest Day  : {formatted_day} ({busiest_day_messages} messages)")

    print(
        f"Busiest Hour : {busiest_hour:02d}:00 - "
        f"{busiest_hour+1:02d}:00 "
        f"(avg {avg_messages:.1f} messages/day)"
    )

    print("=" * 60)



In [95]:
busiest_day_hour(messages)


MOST ACTIVE DAY AND HOUR
Busiest Day  : 30 March 2026 (165 messages)
Busiest Hour : 09:00 - 10:00 (avg 7.6 messages/day)


In [96]:
import numpy as np
def build_activity_matrix(messages):

    # Get participant names
    participants = sorted(list(set(msg["sender"] for msg in messages)))

    # Map each participant to a row index
    person_index = {}

    for i, person in enumerate(participants):
        person_index[person] = i

    # Create 6 × 24 matrix
    activity = np.zeros((len(participants), 24), dtype=int)

    # Fill matrix
    for msg in messages:

        person = msg["sender"]

        hour = int(msg["time"].split(":")[0])

        row = person_index[person]

        activity[row][hour] += 1

    return activity, participants

In [97]:
activity_matrix, participants = build_activity_matrix(messages)

print(activity_matrix)

[[  0   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0]
 [  0  48  61  30   7  45  17  26  45 173 183  78  26   0   0   0   0   0
    0   0   0   0   0   0]
 [  0  48  45  35   8  20  10  30  46 149 164  84  37   0   0   0   0   0
    0   0   0   0   0   0]
 [  0  10  40  35   6  54  10  15  44 140  68  35  16   0   0   0   0   0
    0   0   0   0   0   0]
 [  0  63  66  87   9  50  23  61  56 210 198  92  29   0   0   0   0   0
    0   0   0   0   0   0]]


In [98]:
def print_heatmap(activity_matrix, participants):

    print("=" * 90)
    print("CHAT ACTIVITY HEATMAP")
    print("=" * 90)

    # Hour Header
    print("Name      ", end="")

    for h in range(24):
        print(f"{h:02}", end=" ")

    print()

    for i in range(len(participants)):

        print(f"{participants[i]:<10}", end="")

        maximum = np.max(activity_matrix[i])

        for value in activity_matrix[i]:

            if maximum == 0:
                symbol = ". "

            else:

                ratio = value / maximum

                if ratio == 0:
                    symbol = ". "

                elif ratio <= 0.25:
                    symbol = "░ "

                elif ratio <= 0.50:
                    symbol = "▒ "

                else:
                    symbol = "█ "

            print(symbol, end="")

        print()

In [99]:
print_heatmap(activity_matrix, participants)

CHAT ACTIVITY HEATMAP
Name      00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Game      . █ . . . . . . . . . . . . . . . . . . . . . . 
Kunal Sgt . ▒ ▒ ░ ░ ░ ░ ░ ░ █ █ ▒ ░ . . . . . . . . . . . 
Naman 2   . ▒ ▒ ░ ░ ░ ░ ░ ▒ █ █ █ ░ . . . . . . . . . . . 
Rahul     . ░ ▒ ░ ░ ▒ ░ ░ ▒ █ ▒ ░ ░ . . . . . . . . . . . 
~HarshS Sgt. ▒ ▒ ▒ ░ ░ ░ ▒ ▒ █ █ ▒ ░ . . . . . . . . . . . 


In [100]:
stop_words = {
    "i", "is", "the", "a", "an", "and", "or", "to",
    "of", "in", "on", "for", "at", "it", "its",
    "this", "that", "with", "be", "are", "am",
    "was", "were", "you", "your", "my", "me",
    "we", "our", "they", "their", "he", "she",
    "him", "her", "them", "have", "has", "had",
    "do", "does", "did", "will", "would", "can",
    "could", "should", "not", "but", "if", "so",
    "im", "i'm", "its", "it's"
}

In [101]:
def top_words(messages):

    word_count = {}

    punctuation = ".,!?;:'\"()[]{}<>-/\\@#$%^&*_+=|`~"

    for msg in messages:

        text = msg["message"]

        # Skip media and deleted messages
        if text == "<Media omitted>":
            continue

        if text == "This message was deleted":
            continue

        words = text.split()

        for word in words:

            word = word.lower()

            word = word.strip(punctuation)

            if word == "":
                continue

            if word in stop_words:
                continue

            if word in word_count:
                word_count[word] += 1
            else:
                word_count[word] = 1

    top10 = sorted(
        word_count.items(),
        key=lambda x: x[1],
        reverse=True
    )[:10]

    print("=" * 65)
    print("TOP 10 MOST USED WORDS")
    print("=" * 65)

    maximum = top10[0][1]

    for word, count in top10:

        bar = "█" * int((count / maximum) * 25)

        print(f"{word:<15} {count:>5}  {bar}")

In [102]:
top_words(messages)

TOP 10 MOST USED WORDS
h                 257  █████████████████████████
nhi               180  █████████████████
toh               153  ██████████████
sgt⁩              153  ██████████████
hai               147  ██████████████
m                 118  ███████████
😂                 117  ███████████
⁨naman            111  ██████████
2⁩                111  ██████████
kya               100  █████████


#Feature -6

In [105]:
from datetime import datetime

def response_speed(messages):

    response_times = {}

    for i in range(1, len(messages)):

        prev = messages[i - 1]
        curr = messages[i]

        # Ignore consecutive messages from same sender
        if prev["sender"] == curr["sender"]:
            continue

        # Remove hidden Unicode spaces
        time1 = prev["time"].replace("\u202f", " ").replace("\xa0", " ")
        time2 = curr["time"].replace("\u202f", " ").replace("\xa0", " ")

        # Parse timestamps
        t1 = datetime.strptime(
            prev["date"] + ", " + time1,
            "%d/%m/%y, %I:%M:%S %p"
        )

        t2 = datetime.strptime(
            curr["date"] + ", " + time2,
            "%d/%m/%y, %I:%M:%S %p"
        )

        gap = (t2 - t1).total_seconds() / 60

        sender = curr["sender"]

        if sender not in response_times:
            response_times[sender] = []

        response_times[sender].append(gap)

    average = {}

    for person in response_times:
        average[person] = sum(response_times[person]) / len(response_times[person])

    fastest = min(average, key=average.get)
    slowest = max(average, key=average.get)

    print("=" * 65)
    print("RESPONSE PATTERNS")
    print("=" * 65)

    print(f"Fastest Replier : {fastest} ({average[fastest]:.2f} minutes)")

    if average[slowest] < 60:
        print(f"Slowest Replier : {slowest} ({average[slowest]:.2f} minutes)")
    else:
        print(f"Slowest Replier : {slowest} ({average[slowest] / 60:.2f} hours)")

    return average

In [106]:
response_avg = response_speed(messages)

RESPONSE PATTERNS
Fastest Replier : Kunal Sgt (54.20 minutes)
Slowest Replier : Game (4912.88 hours)


In [107]:
def silent_streak(messages):

    person_dates = {}
    all_dates = set()

    # Store active dates
    for msg in messages:

        person = msg["sender"]

        d = datetime.strptime(
            msg["date"],
            "%d/%m/%y"
        ).date()

        all_dates.add(d)

        if person not in person_dates:
            person_dates[person] = set()

        person_dates[person].add(d)

    start = min(all_dates)
    end = max(all_dates)

    streaks = {}

    print()
    print("=" * 65)
    print("LONGEST SILENT STREAKS")
    print("=" * 65)

    for person in sorted(person_dates):

        longest = 0
        current = 0

        streak_start = None
        best_start = None
        best_end = None

        day = start

        while day <= end:

            if day not in person_dates[person]:

                if current == 0:
                    streak_start = day

                current += 1

                if current > longest:

                    longest = current
                    best_start = streak_start
                    best_end = day

            else:

                current = 0

            day += timedelta(days=1)

        streaks[person] = longest

        if longest == 0:

            print(f"{person:<10}: 0 days (never went silent)")

        elif longest == 1:

            print(f"{person:<10}: 1 day")

        elif best_start is not None:

            print(
                f"{person:<10}: {longest} days "
                f"({best_start.strftime('%d %b')} "
                f"to {best_end.strftime('%d %b')})"
            )

        else:

            print(f"{person:<10}: {longest} days")

    return streaks

In [108]:

silent_streaks = silent_streak(messages)


LONGEST SILENT STREAKS
Game      : 205 days (24 Jul to 13 Feb)
Kunal Sgt : 204 days (25 Jul to 13 Feb)
Naman 2   : 205 days (24 Jul to 13 Feb)
Rahul     : 206 days (24 Jul to 14 Feb)
~HarshS Sgt: 205 days (24 Jul to 13 Feb)


In [41]:
response_avg = response_speed(messages)
silent_streaks = silent_streak(messages)

RESPONSE PATTERNS
Fastest Replier : Rahul (34.95 minutes)
Slowest Replier : Aman (55.36 minutes)

LONGEST SILENT STREAKS
Aman      : 0 days (never went silent)
Karan     : 0 days (never went silent)
Neha      : 0 days (never went silent)
Priya     : 0 days (never went silent)
Rahul     : 0 days (never went silent)
Vikas     : 11 days (23 Apr to 03 May)


# Feature-7


In [109]:

#caring words

caring_words = {
    "take","care","eat","drink","sleep",
    "please","hope","safe","water","home",
    "study","studies","breakfast","dinner",
    "okay","help","rest"
}

Spam

In [110]:
#spammer in tet file


def spam_score(person, messages):

    longest = 1
    current = 1

    for i in range(1, len(messages)):

        if messages[i]["sender"] == person and messages[i-1]["sender"] == person:
            current += 1
            longest = max(longest, current)
        else:
            current = 1

    return longest

Night owl


In [111]:
# Nigt Owl (text in night)

def night_percentage(person, activity_matrix, participants):

    row = participants.index(person)

    total = np.sum(activity_matrix[row])

    if total == 0:
        return 0

    night = (
        np.sum(activity_matrix[row,23:24]) +
        np.sum(activity_matrix[row,0:5])
    )

    return (night/total)*100

average

In [112]:
#Average Words

def average_words(person, messages):

    total = 0
    count = 0

    for msg in messages:

        if msg["sender"] == person:

            total += len(msg["message"].split())
            count += 1

    if count == 0:
        return 0

    return total/count

Drama Score

In [113]:
#Score drama in text

def drama_score(person, messages):

    score = 0

    for msg in messages:

        if msg["sender"] == person:

            text = msg["message"]

            score += text.count("!")

            if text.isupper():
                score += 3

    return score

caring partner

In [114]:
#caring partner

def caring_percentage(person, messages):

    caring = 0
    total = 0

    for msg in messages:

        if msg["sender"] == person:

            words = msg["message"].lower().split()

            total += len(words)

            for word in words:

                if word in caring_words:
                    caring += 1

    if total == 0:
        return 0

    return (caring/total)*100

personality detection

In [120]:
#personality detecetion function

def personality_archetypes(messages,
                           activity_matrix,
                           participants,
                           silent_streaks):

    print("="*70)
    print("PERSONALITY ARCHETYPES")
    print("="*70)

    for person in participants:

        spam = spam_score(person, messages)

        night = night_percentage(
            person,
            activity_matrix,
            participants
        )

        story = average_words(person, messages)

        drama = drama_score(person, messages)

        caring = caring_percentage(person, messages)

        ghost = silent_streaks.get(person,0)

        # Priority Rules

        if night >= 75:

            print(
                f"{person:<10} → THE NIGHT OWL "
                f"({night:.1f}% night messages)"
            )


        elif spam >= 5:

            print(
                f"{person:<10} → THE SPAMMER "
                f"({spam} msgs in a row)"
            )

        elif story >= 20:

            print(
                f"{person:<10} → THE STORYTELLER "
                f"({story:.1f} words/message)"
            )

        elif drama >= 50:

            print(
                f"{person:<10} → THE DRAMA QUEEN "
                f"({drama} drama points)"
            )
        elif ghost >= 10:

            print(
                f"{person:<10} → THE GHOST "
                f"({ghost} silent days)"
            )

        else:

            print(
                f"{person:<10} → THE GROUP MOM "
                f"({caring:.1f}% caring words)"
            )

In [116]:
personality_archetypes(
    messages,
    activity_matrix,
    participants,
    silent_streaks
)

PERSONALITY ARCHETYPES
Game       → THE NIGHT OWL (100.0% night messages)
Kunal Sgt  → THE GHOST (204 silent days)
Naman 2    → THE GHOST (205 silent days)
Rahul      → THE GHOST (206 silent days)
~HarshS Sgt → THE GHOST (205 silent days)


In [117]:
# ==============================================================
# FEATURE 8 : FINAL GROUPDNA REPORT
# This function combines all previous features into one report.
# ==============================================================

from datetime import datetime

def final_report(messages,
                 participants,
                 system_count,
                 media_count,
                 deleted_count,
                 activity_matrix,
                 silent_streaks):

    # ==========================================================
    # Calculate Overall Statistics
    # ==========================================================

    total_messages = len(messages)

    # Dictionary to store messages sent by each participant
    message_count = {}

    for msg in messages:

        sender = msg["sender"]

        if sender in message_count:
            message_count[sender] += 1
        else:
            message_count[sender] = 1


    # ==========================================================
    # Calculate Date Range
    # ==========================================================

    dates = []

    for msg in messages:

        dates.append(
            datetime.strptime(msg["date"], "%d/%m/%y")
        )

    start_date = min(dates)
    end_date = max(dates)

    total_days = (end_date - start_date).days + 1


    # ==========================================================
    # Find Busiest Day
    # ==========================================================

    day_count = {}

    for msg in messages:

        date = msg["date"]

        if date in day_count:
            day_count[date] += 1
        else:
            day_count[date] = 1

    busiest_day = max(day_count, key=day_count.get)


    # ==========================================================
    # Find Busiest Hour
    # ==========================================================

    hour_count = {}

    for msg in messages:

        hour = int(msg["time"].split(":")[0])

        if hour in hour_count:
            hour_count[hour] += 1
        else:
            hour_count[hour] = 1

    busiest_hour = max(hour_count, key=hour_count.get)


    # ==========================================================
    # Sort Participants by Message Count
    # ==========================================================

    sorted_people = sorted(
        message_count.items(),
        key=lambda x: x[1],
        reverse=True
    )


    # ==========================================================
    # REPORT HEADER
    # ==========================================================

    print("\n")
    print("╔" + "═"*70 + "╗")
    print("║{:^70}║".format("GROUPDNA ANALYTICS REPORT"))
    print("╚" + "═"*70 + "╝")

    print()

    print(f"{'Group Name':<20}: Hostel Bois 4ever")
    print(f"{'Period':<20}: {start_date.strftime('%d %B %Y')}  →  {end_date.strftime('%d %B %Y')}")
    print(f"{'Duration':<20}: {total_days} Days")
    print(f"{'Participants':<20}: {len(participants)}")
    print(f"{'Total Messages':<20}: {total_messages:,}")
    print(f"{'System Messages':<20}: {system_count}")
    print(f"{'Media Messages':<20}: {media_count}")
    print(f"{'Deleted Messages':<20}: {deleted_count}")



    # ==========================================================
    # MESSAGE DISTRIBUTION
    # ==========================================================

    print("\n")
    print("="*72)
    print("MESSAGE DISTRIBUTION")
    print("="*72)

    highest = sorted_people[0][1]

    for person, count in sorted_people:

        percentage = (count / total_messages) * 100

        bar = "█" * int((count / highest) * 25)

        print(f"{person:<10} {count:>5} ({percentage:5.1f}%)  {bar}")


    # ==========================================================
    # GROUP ACTIVITY
    # ==========================================================

    print("\n")
    print("="*72)
    print("GROUP ACTIVITY")
    print("="*72)

    print(
        f"Busiest Day  : "
        f"{datetime.strptime(busiest_day,'%d/%m/%y').strftime('%d %B %Y')}"
    )

    print(
        f"Busiest Hour : "
        f"{busiest_hour:02d}:00 - {busiest_hour+1:02d}:00"
    )


    # ==========================================================
    # RESPONSE PATTERNS
    # ==========================================================

    print("\n")
    print("="*72)
    print("RESPONSE PATTERNS")
    print("="*72)

    response_speed(messages)


    # ==========================================================
    # SILENT STREAKS
    # ==========================================================

    print("\n")
    print("="*72)
    print("LONGEST SILENT STREAKS")
    print("="*72)

    silent_streak(messages)


    # ==========================================================
    # TOP WORDS
    # ==========================================================

    print("\n")
    print("="*72)
    print("TOP WORDS")
    print("="*72)

    top_words(messages)


    # ==========================================================
    # CHAT HEATMAP
    # ==========================================================

    print("\n")
    print("="*72)
    print("ACTIVITY HEATMAP")
    print("="*72)

    print_heatmap(activity_matrix, participants)


    # ==========================================================
    # PERSONALITY ARCHETYPES
    # ==========================================================

    print("\n")
    print("="*72)
    print("PERSONALITY ARCHETYPES")
    print("="*72)

    personality_archetypes(
        messages,
        activity_matrix,
        participants,
        silent_streaks
    )


    # ==========================================================
    # FOOTER
    # ==========================================================

    print("\n")
    print("╔" + "═"*70 + "╗")
    print("║{:^70}║".format("END OF GROUPDNA REPORT"))
    print("╚" + "═"*70 + "╝")

In [118]:
# Generate Silent Streak Dictionary
silent_streaks = silent_streak(messages)

# Generate Final Report
final_report(
    messages,
    participants,
    system_count,
    media_count,
    deleted_count,
    activity_matrix,
    silent_streaks
)


LONGEST SILENT STREAKS
Game      : 205 days (24 Jul to 13 Feb)
Kunal Sgt : 204 days (25 Jul to 13 Feb)
Naman 2   : 205 days (24 Jul to 13 Feb)
Rahul     : 206 days (24 Jul to 14 Feb)
~HarshS Sgt: 205 days (24 Jul to 13 Feb)


╔══════════════════════════════════════════════════════════════════════╗
║                      GROUPDNA ANALYTICS REPORT                       ║
╚══════════════════════════════════════════════════════════════════════╝

Group Name          : Hostel Bois 4ever
Period              : 24 July 2025  →  29 June 2026
Duration            : 341 Days
Participants        : 5
Total Messages      : 2,833
System Messages     : 1
Media Messages      : 0
Deleted Messages    : 2


MESSAGE DISTRIBUTION
~HarshS Sgt   944 ( 33.3%)  █████████████████████████
Kunal Sgt    739 ( 26.1%)  ███████████████████
Naman 2      676 ( 23.9%)  █████████████████
Rahul        473 ( 16.7%)  ████████████
Game           1 (  0.0%)  


GROUP ACTIVITY
Busiest Day  : 30 March 2026
Busiest Hour : 09:00 - 